In [20]:
from datasets import load_dataset

# load the entire dataset (returns a DatasetDict containing all splits)
dataset = load_dataset("kurry/sp500_earnings_transcripts", split="train")

print(dataset)

Dataset({
    features: ['symbol', 'quarter', 'year', 'date', 'content', 'structured_content', 'company_name', 'company_id'],
    num_rows: 33362
})


In [21]:
# selecting the these columns and dropping the heavy ones
light_ds = dataset.select_columns(["symbol", "company_name", "date", "year", "quarter"])

print(light_ds.features)

{'symbol': Value('string'), 'company_name': Value('string'), 'date': Value('string'), 'year': Value('int64'), 'quarter': Value('int64')}


In [22]:
TICKERS = {"WMT", "COST", "HD", "TGT", "M", "KSS", "GPS", "GAP", "BBY", "BBBY", "JCP"}
START = "2015-01-01"
END = "2025-12-31 23:59:59"

def keep_row(batch):
    return [
        sym in TICKERS and date is not None and START <= date <= END
        for sym, date in zip(batch["symbol"], batch["date"])
    ]

filtered = light_ds.filter(keep_row, batched=True)

print(filtered)

Dataset({
    features: ['symbol', 'company_name', 'date', 'year', 'quarter'],
    num_rows: 263
})


In [23]:
import pandas as pd

df = pd.DataFrame(filtered)

In [24]:
df["symbol"].value_counts()

symbol
BBY     41
COST    41
HD      41
TGT     40
WMT     36
KSS     25
M       25
BBBY    14
Name: count, dtype: int64

In [25]:
zero_set = TICKERS - set(df["symbol"])
print(zero_set)

{'GPS', 'JCP', 'GAP'}


In [26]:
# first and last call date 
first_last_call = df.groupby("symbol")["date"].agg(["min", "max"])
print(first_last_call)

                        min                  max
symbol                                          
BBBY    2015-01-08 17:00:00  2018-04-12 17:00:00
BBY     2015-03-03 08:00:00  2025-03-04 08:00:00
COST    2015-03-05 11:00:00  2025-03-06 17:00:00
HD      2015-02-24 09:00:00  2025-02-25 09:00:00
KSS     2015-02-26 08:30:00  2021-03-02 09:00:00
M       2015-02-24 10:30:00  2021-02-23 08:00:00
TGT     2015-02-25 10:30:00  2025-03-04 09:00:00
WMT     2015-02-19 07:00:00  2025-02-20 08:00:00


In [27]:
# is the GAP really missing or stored under a different name
# search the FULL light_ds (not `filtered` - that only contains our TICKERS, so it can't find a different symbol)
full_df = light_ds.to_pandas()  # light columns only, so this is cheap

# \b = word boundary, so "Gap" matches "The Gap, Inc." but not "Singapore"
for pattern in [r"\bGap\b", r"Penney"]:
    # case=False -> ignore upper/lower case; na=False -> rows with a missing company_name count as "no match" instead of erroring
    matches = full_df[full_df["company_name"].str.contains(pattern, case=False, na=False)]
    print(f"--- '{pattern}' ---")
    if matches.empty:
        print("no matches in the whole dataset")
    else:
        print(matches.groupby(["symbol", "company_name"])["date"].agg(["count", "min", "max"]))

--- '\bGap\b' ---
no matches in the whole dataset
--- 'Penney' ---
no matches in the whole dataset


In [28]:
# which columns have missing values? (this is why DataFrame.value_counts() showed 199 instead of 263)
print("missing values in our filtered rows:")
print(df.isna().sum())

print("\nmissing values in the whole dataset:")
print(full_df.isna().sum())

# show a few of the rows with a missing value in our filtered set, to see what they look like
print("\nexample rows with a missing value:")
print(df[df.isna().any(axis=1)].head())

missing values in our filtered rows:
symbol           0
company_name    64
date             0
year             0
quarter          0
dtype: int64

missing values in the whole dataset:
symbol             0
company_name    8985
date               0
year               0
quarter            0
dtype: int64

example rows with a missing value:
    symbol company_name                 date  year  quarter
199    KSS         None  2015-02-26 08:30:00  2014        4
200    KSS         None  2016-02-25 08:30:00  2015        4
201    KSS         None  2015-11-12 08:30:00  2015        3
202    KSS         None  2015-08-13 08:30:00  2015        2
203    KSS         None  2015-05-14 08:30:00  2015        1
